Import libraries.

In [2]:
import os
os.chdir('..')

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

from modules.data_setup import create_specific_dataset, sports_dataframe
from modules.training import run_training
from modules.architecture import Conv
from modules.utils import nb_parameters

In [13]:
seed = 42 #DON'T CHANGE THIS VALUE

### Preparations of dataset and subsets: 

We will work with gray and then with colored data to compare the impact of color and also to reduce de number of parametters by 3 for the gray images. Our hypothesis is that for sport comparation, color will be usefull so improve accuracy.

Functions used can be found in `data_setup.py`.

#### 3 fairly different classes

In [3]:
DIFF_CLASSES_3 = ["football", "nascar racing", "olympic wrestling"]
print(f"For DIFF_CLASSES_3: Loading {len(DIFF_CLASSES_3)} classes: {DIFF_CLASSES_3}...")
train_ds_Gray_DIFF_3, val_ds_Gray_DIFF_3, test_ds_Gray_DIFF_3 = create_specific_dataset(DIFF_CLASSES_3, "Gray")
train_ds_Color_DIFF_3, val_ds_Color_DIFF_3, test_ds_Color_DIFF_3 = create_specific_dataset(DIFF_CLASSES_3, "Color")

For DIFF_CLASSES_3: Loading 3 classes: ['football', 'nascar racing', 'olympic wrestling']...


#### 3 fairly similar classes

In [4]:
SIM_CLASSES_3 = ["football", "rugby", "ampute football"]
print(f"For SIM_CLASSES_3: Loading {len(SIM_CLASSES_3)} classes: {SIM_CLASSES_3}...")
train_ds_Gray_SIM_3, val_ds_Gray_SIM_3, test_ds_Gray_SIM_3 = create_specific_dataset(SIM_CLASSES_3, "Gray")
train_ds_Color_SIM_3, val_ds_Color_SIM_3, test_ds_Color_SIM_3 = create_specific_dataset(SIM_CLASSES_3, "Color")

For SIM_CLASSES_3: Loading 3 classes: ['football', 'rugby', 'ampute football']...


#### 6 fairly different classes

In [5]:
DIFF_CLASSES_6 = ["football", "nascar racing", "olympic wrestling", "surfing", "archery", "arm wrestling"]
print(f"For DIFF_CLASSES_6: Loading {len(DIFF_CLASSES_6)} classes: {DIFF_CLASSES_6}...")
train_ds_Gray_DIFF_6, val_ds_Gray_DIFF_6, test_ds_Gray_DIFF_6 = create_specific_dataset(DIFF_CLASSES_6, "Gray")
train_ds_Color_DIFF_6, val_ds_Color_DIFF_6, test_ds_Color_DIFF_6 = create_specific_dataset(DIFF_CLASSES_6, "Color")

For DIFF_CLASSES_6: Loading 6 classes: ['football', 'nascar racing', 'olympic wrestling', 'surfing', 'archery', 'arm wrestling']...


#### 12 classes

In [6]:
CLASSES_12 = ["football", "olympic wrestling", "archery", "arm wrestling", "swimming", "ski jumping", "rowing", "pole dancing", "formula 1 racing", "boxing", "baton twirling", "axe throwing"]
print(f"For CLASSES_12: Loading {len(CLASSES_12)} classes: {CLASSES_12[:5]}...")
train_ds_Gray_12, val_ds_Gray_12, test_ds_Gray_12 = create_specific_dataset(CLASSES_12, "Gray")
train_ds_Color_12, val_ds_Color_12, test_ds_Color_12 = create_specific_dataset(CLASSES_12, "Color")

For CLASSES_12: Loading 12 classes: ['football', 'olympic wrestling', 'archery', 'arm wrestling', 'swimming']...


#### 20 classes

In [7]:
CLASSES_20 = ["football", "nascar racing", "olympic wrestling", "surfing", "archery", "arm wrestling", "swimming", "ampute football", "skydiving", "sky surfing", "ski jumping", "rowing", "pommel horse", "pole dancing", "pole climbing", "formula 1 racing", "boxing", "baton twirling", "axe throwing", "water cycling"]
print(f"For CLASSES_20: Loading {len(CLASSES_20)} classes: {CLASSES_20[:5]}...")
train_ds_Gray_20, val_ds_Gray_20, test_ds_Gray_20 = create_specific_dataset(CLASSES_20, "Gray")
train_ds_Color_20, val_ds_Color_20, test_ds_Color_20 = create_specific_dataset(CLASSES_20, "Color")

For CLASSES_20: Loading 20 classes: ['football', 'nascar racing', 'olympic wrestling', 'surfing', 'archery']...


#### 100 classes (whole dataset)

In [8]:
CLASSES_100 = sports_dataframe['labels'].unique().tolist()
print(f"For CLASSES_100: Loading {len(CLASSES_100)} classes: {CLASSES_100[:5]}...")
train_ds_Gray_100, val_ds_Gray_100, test_ds_Gray_100 = create_specific_dataset(CLASSES_100, "Gray")
train_ds_Color_100, val_ds_Color_100, test_ds_Color_100 = create_specific_dataset(CLASSES_100, "Color")

For CLASSES_100: Loading 100 classes: ['air hockey', 'ampute football', 'archery', 'arm wrestling', 'axe throwing']...


### Dataset Analysis
Here is the dataset analysis (i let you do it alvaro) I just printed number of classes and of images for each types of datasets.

In [9]:
num_classes = len(train_ds_Gray_100.classes)
print("Classes :", num_classes)
print("Train :", len(train_ds_Gray_100), "| Val :", len(val_ds_Gray_100), "| Test :", len(test_ds_Gray_100))

Classes : 100
Train : 13492 | Val : 500 | Test : 500


### Disclaimer

- Model Net with activation function, regularization and convolution. We set convolution by default to None, if so, it just flat the image. This + `class Conv` can be found in `modules\architecture.py`

- Optimizers, schedulers and other small functions can be found in `modules\utils.py`

- Finally, the `train` and `run_training` functions can be found in `modules\training.py`

### Temporary random tests

In [10]:
def plot_metrics(metrics, cm_epochs=()):
    epochs = range(1, len(metrics["loss_train"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, metrics["accuracy_train"], label="Train")
    axes[0].plot(epochs, metrics["accuracy_val"], label="Validation")
    axes[0].set_title("Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(epochs, metrics["loss_train"], label="Train")
    axes[1].plot(epochs, metrics["loss_val"], label="Validation")
    axes[1].set_title("Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    for epoch in cm_epochs:
        index = epoch - 1
        if index < 0 or index >= len(metrics["confusion_matrices"]):
            print("No confusion matrix for epoch", epoch)
            continue

        cm = metrics["confusion_matrices"][index]
        plt.figure(figsize=(8, 8))
        plt.imshow(cm, cmap="Blues")
        plt.colorbar()
        plt.title("Confusion matrix - epoch " + str(epoch))
        plt.xlabel("Predicted class")
        plt.ylabel("True class")
        plt.show()

### Experiments

Let's start our process with default parameters. For the activation function we will use "relu". We will not use any regularizations, not any schedulers and for the optimizer we will use SGD by default for now.

For the convolution, we use first no convolution at all even we now this is a bad idea. It mean we will just flat the images and give it to the model.

Let's arbitrarily set the learning rate to 0.001 and the number of epochs to 20 for now. Our first objective is to find the best hiddens layers configurations to use it for the rest of the process.
Let's first work with the set of 20 sport (with somes differents and some similar)

Note : In the idea, our results should be the worst at the beginning of the process and the best at the end.

In [ ]:
metrics_Gray_12c_256h128h, net_Gray_12c_256h128h = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[256, 128], epochs=20, lr=0.001, seed=seed)
# plot_metrics(metrics_Gray_12c_256h128h, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_256h128h)))

Epoch [1/20], Train Loss: 2.7708, Val Loss: 2.5951, Val Acc: 0.0833
Epoch [2/20], Train Loss: 2.4672, Val Loss: 2.5480, Val Acc: 0.1000
Epoch [3/20], Train Loss: 2.4002, Val Loss: 2.4969, Val Acc: 0.2000
Epoch [4/20], Train Loss: 2.3336, Val Loss: 2.3800, Val Acc: 0.2333
Epoch [5/20], Train Loss: 2.2522, Val Loss: 2.3196, Val Acc: 0.2000
Epoch [6/20], Train Loss: 2.2158, Val Loss: 2.3018, Val Acc: 0.2000
Epoch [7/20], Train Loss: 2.1698, Val Loss: 2.3171, Val Acc: 0.1500
Epoch [8/20], Train Loss: 2.1443, Val Loss: 2.2634, Val Acc: 0.1833
Epoch [9/20], Train Loss: 2.1011, Val Loss: 2.2034, Val Acc: 0.2833
Epoch [10/20], Train Loss: 2.0556, Val Loss: 2.2631, Val Acc: 0.2500
Epoch [11/20], Train Loss: 2.0183, Val Loss: 2.2491, Val Acc: 0.1833
Epoch [12/20], Train Loss: 2.0015, Val Loss: 2.2161, Val Acc: 0.2333
Epoch [13/20], Train Loss: 1.9836, Val Loss: 2.1885, Val Acc: 0.2500
Epoch [14/20], Train Loss: 1.9353, Val Loss: 2.2433, Val Acc: 0.2000
Epoch [15/20], Train Loss: 1.9188, Val Loss

In [ ]:
metrics_Gray_12c_128h64h, net_Gray_12c_128h64h= run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001)
# plot_metrics(metrics_Gray_12c_128h64h, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_128h64h)))

In [ ]:
metrics_Gray_12c_128h, net_Gray_12c_128h = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128], epochs=20, lr=0.001)
# plot_metrics(metrics_Gray_12c_128h, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_128h)))

In [ ]:
metrics_Gray_12c_512h_128h, net_Gray_12c_512h_128h = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[512, 128], epochs=20, lr=1e-3)
# plot_metrics(metrics_Gray_12c_512h_128h, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_512h_128h)))

In [ ]:
metrics_Gray_12c_64h_32h, net_Gray_12c_64h_32h = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[64, 32], epochs=20, lr=1e-3)
# plot_metrics(metrics_Gray_12c_64h_32h, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_64h_32h)))

In [ ]:
metrics_Gray_12c_128h_64h_32h, net_Gray_12c_128h_64h_32h = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64, 32], epochs=20, lr=1e-3)
# plot_metrics(metrics_Gray_12c_128h_64h_32h, [10, 20])
print("\nNumber of parameters :" + str(nb_parameters(net_Gray_12c_128h_64h_32h)))

Comparing differents results and not focucing on overfitting of ran, best results are gets with x hiddens layers of 256 and 128 neurons which represent 12 880 788 parameters. But we choosed to use 128 and 64 which is a bit less efficient but represent the half of parameters. It will be easier then, so we will use it for the next of the process.

Let's add now a 2D convolution. As we don't know what is the best configuration (well of course we have theorical ideas but we need to check in practice), we will try with differents kernel sizes (3 and 7). We also need to determine the number of output channel of the convolution which is the number of filter of the convolution. We will try 3 arbitral values : 8, 16 and 32. It represent 6 ran.
Let's do it

In [ ]:
metrics_Gray_12c_8o_3k_poolmax, net_Gray_12c_8o_3k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
# plot_metrics(metrics_Gray_12c_8o_3k_poolmax, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_8o_3k_poolmax)))

In [ ]:
metrics_Gray_12c_16o_3k_poolmax, net_Gray_12c_16o_3k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=16, kernel_size=3, pooling="max")
# plot_metrics(metrics_Gray_12c_16o_3k_poolmax, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_16o_3k_poolmax)))

In [ ]:
metrics_Gray_12c_32o_3k_poolmax, net_Gray_12c_32o_3k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=32, kernel_size=3, pooling="max")
# plot_metrics(metrics_Gray_12c_32o_3k_poolmax, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_32o_3k_poolmax)))

In [ ]:
metrics_Gray_12c_8o_7k_poolmax, net_Gray_12c_8o_7k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=7, pooling="max")
# plot_metrics(metrics_Gray_12c_8o_7k_poolmax, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_8o_7k_poolmax)))

In [ ]:
metrics_Gray_12c_16o_7k_poolmax, net_Gray_12c_16o_7k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=16, kernel_size=7, pooling="max")
# plot_metrics(metrics_Gray_12c_16o_7k_poolmax, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_16o_7k_poolmax)))

In [ ]:
metrics_Gray_12c_32o_7k_poolmax, net_Gray_12c_32o_7k_poolmax = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=32, kernel_size=7, pooling="max")
# plot_metrics(metrics_Gray_12c_32o_7k_poolmax, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_32o_7k_poolmax)))

Best results are get with kernel dimmensions of 3. And the more filter we have, the more accurate is the model. But since the number of parameters explode with the number of filter, we'll keep working with 8 filters.

Let's compare it with average pooling. 

In [ ]:
metrics_Gray_12c_8o_3k_poolaverage, net_Gray_12c_8o_3k_poolaverage = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="average")
# plot_metrics(metrics_Gray_12c_8o_3k_poolaverage, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_32o_3k_poolaverage)))

As expected, poolmax is better than poolaverage.

Until here, we used relu because we know by theoritical that this is the best but we should compare the results between the two others activations functions

In [ ]:
metrics_Gray_12_tanh, net_Gray_12c_tanh = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], activation_function="tanh", epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
# plot_metrics(metrics_Gray_12_tanh, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_tanh)))

In [ ]:
metrics_Gray_12c_sigmoid, net_Gray_12c_sigmoid = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], activation_function="sigmoid", epochs=20, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
# plot_metrics(metrics_Gray_12c_sigmoid, [10, 20])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_sigmoid)))

Here as we see, relu still be the best option. Now, let's focus on the loss computation to check model behaviour

In [ ]:
metrics_Gray_12c, net_Gray_12c = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
plot_metrics(metrics_Gray_12c, [])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c)))

We see that the model is overfitting. We then need to add some regularization to prevent our modele to to so. Let's try with weight decay and 0.0001 penalty 

In [ ]:
metrics_Gray_12c_weight_decay, net_Gray_12c_weight_decay = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.0001, epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
plot_metrics(metrics_Gray_12c_weight_decay, [])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_weight_decay)))

In [ ]:
metrics_Gray_12c_weight_decay, net_Gray_12c_weight_decay = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.001, epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
plot_metrics(metrics_Gray_12c_weight_decay, [])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_weight_decay)))

In [ ]:
metrics_Gray_12c_weight_decay, net_Gray_12c_weight_decay = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.01, epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
plot_metrics(metrics_Gray_12c_weight_decay, [])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_weight_decay)))

In [ ]:
metrics_Gray_12c_weight_decay, net_Gray_12c_weight_decay = run_training(train_ds_Gray_12, val_ds_Gray_12, num_classes=20, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.1, epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=1, out_channels=8, kernel_size=3, pooling="max")
plot_metrics(metrics_Gray_12c_weight_decay, [])
print("\nNumber of parameters : " + str(nb_parameters(net_Gray_12c_weight_decay)))

Overfitting is now fixed. Now, let's run our model with color. In theory, accurate results should be better. We also want to check if the model do not overfit

In [ ]:
metrics_Color_12c, net_Color_12c = run_training(train_ds_Color_12, val_ds_Color_12, num_classes=20, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.1, epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=3, out_channels=8, kernel_size=3, pooling="max")
plot_metrics(metrics_Color_12c, [])
print("\nNumber of parameters : " + str(nb_parameters(net_Color_12c)))

We also should also add better optimizer than SGD to get a better weight computation and schedulers to have the best possible learning rate.

In [ ]:
metrics_Color_12c, net_Color_12c = run_training(train_ds_Color_12, val_ds_Color_12, num_classes=20, hidden_sizes=[128, 64], regularization="weight_decay", reg_param=0.1, optimizer_name="Adam", scheduler_name="step", epochs=40, lr=0.001,
                                                                   convolution=Conv, in_channels=3, out_channels=8, kernel_size=3, pooling="max")
plot_metrics(metrics_Color_12c, [])
print("\nNumber of parameters : " + str(nb_parameters(net_Color_12c)))

Well we found parameters that seems to be relevants.